# Dekodowanie liniowe z cech wąskiego gardła

Ten notebook używa **cech wąskiego gardła** modelu DeepMReye
do dekodowania ruchu gałek ocznych za pomocą regresji liniowej.

Model DeepMReye przetwarza dane fMRI przez CNN i produkuje
reprezentację pośrednią zanim wyda predykcję gazowania.
Zamiast korzystać z wyjścia modelu, używamy tych cech jako wejścia do **regresji ridge**.

Podejście:
1. Wczytanie cech wąskiego gardła ze wszystkich 29 podmiotów resting-state
2. Wybranie 200 cech najlepiej skorelowanych z ruchem gałek ocznych
3. Dekodowanie **wewnątrzpodmiotowe** (5-fold CV, RidgeCV)
4. Dekodowanie **międzypodmiotowe** (leave-one-subject-out)
5. Ocena jakości: Pearson r i R²

## Importy

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_1samp, linregress
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.metrics import r2_score

## Wczytanie i inspekcja danych

Plik `dataset7_bottleneck.pkl` zawiera słownik `{nazwa_pliku → bundle}`.
Każdy bundle ma klucze:
- `bottleneck_combined` - macierz `(T, 6144)` z cechami wąskiego gardła
- `bottleneck_left`, `bottleneck_right` - cechy dla lewego i prawego oka osobno
- `true_x`, `true_y` - rzeczywiste współrzędne gazowania (etykiety)
- `T` - liczba punktów czasowych (zwykle 182 per sesję)

In [ ]:
LOAD_PATH = "/mnt/compneuro/deepmreye_finetuning/dataset7_bottleneck.pkl"

with open(LOAD_PATH, "rb") as f:
    data = pickle.load(f)

subjects = list(data.keys())
print(f"Number of files stored: {len(data)}\n")

print(f"{'File':50s} | {'T':>5} | {'b_right':>12} | {'b_left':>12} | {'b_comb':>12}")
for basename, bundle in sorted(data.items()):
    T  = bundle["T"]
    br = bundle["bottleneck_right"].shape
    bl = bundle["bottleneck_left"].shape
    bc = bundle["bottleneck_combined"].shape
    print(f"{basename:50s} | {T:5d} | {str(br):>12} | {str(bl):>12} | {str(bc):>12}")

first_key = sorted(data.keys())[0]
bundle = data[first_key]
print(f"\nKeys in each bundle (example: {first_key})")
for key, val in bundle.items():
    if isinstance(val, np.ndarray):
        print(f"{key:25s}: shape={val.shape}, dtype={val.dtype}, "
              f"min={val.min():.4f}, max={val.max():.4f}, mean={val.mean():.4f}")
    else:
        print(f"{key:25s}: {val}")

## Analiza cech: korelacja z ruchem gałek ocznych

Dla każdego z 29 podmiotów i każdej z 6144 cech obliczamy korelację Pearsona
między wartością cechy a współrzędną gazowania X lub Y.

**Kryterium wyboru cechy:** średnia wartość |r| przez podmioty > próg,
dodatkowo weryfikowana testem t (czy |r| istotnie różni się od 0).

In [ ]:
n_features = 6144

corr_x = np.full((len(subjects), n_features), np.nan)
corr_y = np.full((len(subjects), n_features), np.nan)

for i, subj in enumerate(subjects):
    d = data[subj]
    X = d["bottleneck_combined"]
    tx, ty = d["true_x"], d["true_y"]
    stds = X.std(axis=0)
    active = stds > 1e-6
    for f in np.where(active)[0]:
        corr_x[i, f] = pearsonr(X[:, f], tx)[0]
        corr_y[i, f] = pearsonr(X[:, f], ty)[0]
    print(f"  {i+1}/{len(subjects)}  {subj}")

abs_corr_x = np.abs(corr_x)
abs_corr_y = np.abs(corr_y)
mean_abs_x = np.nanmean(abs_corr_x, axis=0)
mean_abs_y = np.nanmean(abs_corr_y, axis=0)

pval_x = np.ones(n_features)
pval_y = np.ones(n_features)
for f in range(n_features):
    col_x = abs_corr_x[:, f][~np.isnan(abs_corr_x[:, f])]
    col_y = abs_corr_y[:, f][~np.isnan(abs_corr_y[:, f])]
    if len(col_x) > 5:
        _, pval_x[f] = ttest_1samp(col_x, 0)
    if len(col_y) > 5:
        _, pval_y[f] = ttest_1samp(col_y, 0)

print("\nGroup-level features")
for thresh in [0.1, 0.15, 0.2]:
    sig_x = (mean_abs_x > thresh) & (pval_x < 0.05)
    sig_y = (mean_abs_y > thresh) & (pval_y < 0.05)
    print(f"mean|r|>{thresh} + p<0.05: X={np.sum(sig_x)}, Y={np.sum(sig_y)}")

print("\nTop 20 features for X ")
print(f"{'feat':>6}  {'mean|r|':>8}  {'pval':>8}")
for f in np.argsort(mean_abs_x)[::-1][:20]:
    print(f"{f:6d}  {mean_abs_x[f]:8.3f}  {pval_x[f]:8.4f}")

print("\nTop 20 features for Y")
print(f"{'feat':>6}  {'mean|r|':>8}  {'pval':>8}")
for f in np.argsort(mean_abs_y)[::-1][:20]:
    print(f"{f:6d}  {mean_abs_y[f]:8.3f}  {pval_y[f]:8.4f}")

np.save("abs_corr_x_full.npy", abs_corr_x)
np.save("abs_corr_y_full.npy", abs_corr_y)

## Selekcja cech i definicja dekodera

Wybieramy **200 najlepszych cech** dla X i 200 dla Y (osobno).
Cechy muszą być ważne w co najmniej 25 z 29 podmiotów (filtr stabilności).

Funkcja `within_subject_decode`:
- Używa 5-fold cross-validation wewnątrz podmiotu
- **Korekcja znaku**: przed skalowaniem wyrównuje kierunek korelacji każdej cechy
  (cechy negatywnie skorelowane są odwracane, żeby RidgeCV mógł je sumować)
- Skalowanie `StandardScaler` na danych treningowych, transformacja na testowych
- **RidgeCV** - regresja grzbietowa z automatycznym doborem `alpha` przez CV

In [ ]:
abs_corr_x = np.load("abs_corr_x_full.npy")
abs_corr_y = np.load("abs_corr_y_full.npy")

valid_x = np.sum(~np.isnan(abs_corr_x), axis=0) >= 25
valid_y = np.sum(~np.isnan(abs_corr_y), axis=0) >= 25

mean_abs_x = np.where(valid_x, np.nanmean(abs_corr_x, axis=0), 0)
mean_abs_y = np.where(valid_y, np.nanmean(abs_corr_y, axis=0), 0)

top200_x = np.argsort(mean_abs_x)[::-1][:200]
top200_y = np.argsort(mean_abs_y)[::-1][:200]

print(f"Top 10 features for X: {top200_x[:10]}")
print(f"Top 10 features for Y: {top200_y[:10]}")


def within_subject_decode(X_feats, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=False)
    y_pred_all = np.zeros_like(y, dtype=float)

    for train_idx, test_idx in kf.split(X_feats):
        X_train, X_test = X_feats[train_idx], X_feats[test_idx]
        y_train = y[train_idx]

        signs = np.array([
            np.sign(pearsonr(X_train[:, f], y_train)[0])
            if X_train[:, f].std() > 1e-6 else 1.0
            for f in range(X_train.shape[1])
        ])
        signs[signs == 0] = 1.0
        X_train = X_train * signs
        X_test = X_test  * signs

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        clf = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 1000, 10000])
        clf.fit(X_train, y_train)
        y_pred_all[test_idx] = clf.predict(X_test)

    r, _ = pearsonr(y_pred_all, y)
    return r, y_pred_all

## Dekodowanie wewnątrzpodmiotowe

Dla każdego z 29 podmiotów uruchamiamy `within_subject_decode`.
Model jest trenowany i testowany wyłącznie na danych tego samego podmiotu.

In [ ]:
print(f"{'subj':<12} {'r_x':>8} {'r_y':>8}")

all_r_x_ws, all_r_y_ws = [], []
all_preds_ws = {}

for subj in subjects:
    d = data[subj]
    X = d["bottleneck_combined"]
    tx, ty = d["true_x"], d["true_y"]

    r_x, pred_x = within_subject_decode(X[:, top200_x], tx)
    r_y, pred_y = within_subject_decode(X[:, top200_y], ty)

    all_r_x_ws.append(r_x)
    all_r_y_ws.append(r_y)
    all_preds_ws[subj] = {"pred_x": pred_x, "pred_y": pred_y,
                          "true_x": tx, "true_y": ty}
    print(f"{subj:<12} {r_x:>8.3f} {r_y:>8.3f}")

print(f"\n{'mean':<12} {np.mean(all_r_x_ws):>8.3f} {np.mean(all_r_y_ws):>8.3f}")
print(f"{'std':<12} {np.std(all_r_x_ws):>8.3f} {np.std(all_r_y_ws):>8.3f}")
print(f"{'n>0.2':<12} {sum(r>0.2 for r in all_r_x_ws):>8} {sum(r>0.2 for r in all_r_y_ws):>8}")
print(f"{'n>0.3':<12} {sum(r>0.3 for r in all_r_x_ws):>8} {sum(r>0.3 for r in all_r_y_ws):>8}")

### Wykres 1 - Pearson r per podmiot (within-subject)

Niebieski/zielony = r > 0 (pozytywna korelacja), czerwony = r < 0.

Wyniki: `figures\5-Pearson_r_per_podmiot.png`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(subjects))
labels = [s[:4] for s in subjects]

for ax, rs, title, base_color in [
    (axes[0], all_r_x_ws, "X gaze - r per subject (within-subject)", "#378ADD"),
    (axes[1], all_r_y_ws, "Y gaze - r per subject (within-subject)", "#1D9E75"),
]:
    colors = [base_color if r > 0 else "#E24B4A" for r in rs]
    ax.bar(x, rs, color=colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.2, color="red", linewidth=1, linestyle="--", label="r=0.2")
    ax.axhline(0.3, color="orange", linewidth=1, linestyle="--", label="r=0.3")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Pearson r")
    ax.set_ylim(-1, 1)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("./plots/bottleneck_decoding_r_per_subject.png", dpi=150)
plt.show()

### Wykres 2 - Szeregi czasowe: 6 najlepszych podmiotów

Czarna linia = rzeczywiste gazowanie, kolorowa = predykcja modelu.
Wyświetlone są 6 podmiotów z najwyższym Pearson r dla danej osi.

Wyniki: `figures\5-X_gaze_within.png`
Wyniki: `figures\5-Y_gaze_within.png`

In [ ]:
for axis_name, all_r, pred_key, true_key, color in [
    ("X", all_r_x_ws, "pred_x", "true_x", "#378ADD"),
    ("Y", all_r_y_ws, "pred_y", "true_y", "#1D9E75"),
]:
    best6 = sorted(subjects, key=lambda s: all_r[subjects.index(s)])[::-1][:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    axes = axes.flatten()
    for i, subj in enumerate(best6):
        pred = all_preds_ws[subj][pred_key]
        true = all_preds_ws[subj][true_key]
        r = pearsonr(pred, true)[0]
        t = np.arange(len(true))
        axes[i].plot(t, true, color="black", linewidth=1, label="true", alpha=0.8)
        axes[i].plot(t, pred, color=color, linewidth=1, label="pred", alpha=0.8)
        axes[i].set_title(f"{subj[:4]}  r={r:.2f}", fontsize=10)
        axes[i].set_xlabel("timepoint")
        axes[i].set_ylabel(f"gaze {axis_name}")
        axes[i].legend(fontsize=7)
    plt.suptitle(f"{axis_name} gaze — predicted vs true (best 6 subjects)", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"./plots/bottleneck_timeseries_{axis_name.lower()}_best6.png", dpi=150)
    plt.show()

## Wyniki: $R^2$ (współczynnik determinacji)

$R^2$ mierzy jaka część wariancji w rzeczywistym gazowaniu jest wyjaśniona przez predykcję.
Obliczamy go metodą regresji liniowej (true ~ pred), co jest odporne na skalę.
`$R^2$ =$r^2$` gdzie r to współczynnik korelacji Pearsona z tej regresji.

In [ ]:
print("R^2 from linear regression (true ~ pred)")
print(f"{'subj':<12} {'R2_x':>8} {'R2_y':>8} {'R2_xy':>8}")

all_r2_x_ws, all_r2_y_ws, all_r2_xy_ws = [], [], []

for subj in subjects:
    pred_x = all_preds_ws[subj]["pred_x"]
    pred_y = all_preds_ws[subj]["pred_y"]
    true_x = all_preds_ws[subj]["true_x"]
    true_y = all_preds_ws[subj]["true_y"]

    _, _, r_x, _, _ = linregress(pred_x, true_x)
    _, _, r_y, _, _ = linregress(pred_y, true_y)
    r2_x  = r_x ** 2
    r2_y  = r_y ** 2
    r2_xy = (r2_x + r2_y) / 2

    all_r2_x_ws.append(r2_x)
    all_r2_y_ws.append(r2_y)
    all_r2_xy_ws.append(r2_xy)
    print(f"{subj:<12} {r2_x:>8.3f} {r2_y:>8.3f} {r2_xy:>8.3f}")

print(f"\n{'median':<12} {np.median(all_r2_x_ws):>8.3f} {np.median(all_r2_y_ws):>8.3f} {np.median(all_r2_xy_ws):>8.3f}")
print(f"{'mean':<12} {np.mean(all_r2_x_ws):>8.3f} {np.mean(all_r2_y_ws):>8.3f} {np.mean(all_r2_xy_ws):>8.3f}")
print(f"{'std':<12} {np.std(all_r2_x_ws):>8.3f} {np.std(all_r2_y_ws):>8.3f} {np.std(all_r2_xy_ws):>8.3f}")
print(f"{'n>0':<12} {sum(r>0 for r in all_r2_x_ws):>8} {sum(r>0 for r in all_r2_y_ws):>8} {sum(r>0 for r in all_r2_xy_ws):>8}")

### Wykres 3 - $R^2$ per podmiot + rozkład violin

Słupki: R² per podmiot dla X, Y, i łącznie X+Y.
Wykres skrzypcowy: rozkład R² we wszystkich podmiotach.

Wyniki: `figures\5-X_Y_joint_per_subject.png`
Wyniki: `figures\5-r2_violin.png`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x  = np.arange(len(subjects))
labels = [s[:4] for s in subjects]

for ax, rs, title, color in [
    (axes[0], all_r2_x_ws,  "X gaze — R² per subject",    "#378ADD"),
    (axes[1], all_r2_y_ws,  "Y gaze — R² per subject",    "#1D9E75"),
    (axes[2], all_r2_xy_ws, "X+Y joint — R² per subject", "#7F77DD"),
]:
    colors = [color if r > 0 else "#E24B4A" for r in rs]
    ax.bar(x, rs, color=colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.1, color="red", linewidth=1, linestyle="--", label="R²=0.1")
    ax.axhline(0.2, color="orange", linewidth=1, linestyle="--", label="R²=0.2")
    ax.axhline(np.median(rs), color="purple", linewidth=1.5,
               linestyle="-", label=f"median={np.median(rs):.3f}")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("R²")
    ax.set_ylim(-1, 1)
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("./plots/bottleneck_r2_per_subject.png", dpi=150)
plt.show()

# Violin plot
fig, ax = plt.subplots(figsize=(6, 5))
parts = ax.violinplot([all_r2_x_ws, all_r2_y_ws, all_r2_xy_ws],
                      positions=[1, 2, 3], showmedians=True)
colors = ["#378ADD", "#1D9E75", "#7F77DD"]
for pc, color in zip(parts["bodies"], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
parts["cmedians"].set_color("red")
parts["cmedians"].set_linewidth(2)
for i, (rs, color) in enumerate(zip([all_r2_x_ws, all_r2_y_ws, all_r2_xy_ws], colors), 1):
    ax.scatter(np.full(len(rs), i) + np.random.uniform(-0.05, 0.05, len(rs)),
               rs, s=20, color=color, alpha=0.7, zorder=3)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["X gaze", "Y gaze", "X+Y joint"])
ax.set_ylabel("R²")
ax.set_ylim(0, 1)
ax.set_title("R² distribution across subjects\n(red line = median)", fontsize=11)
plt.tight_layout()
plt.savefig("./plots/bottleneck_r2_violin.png", dpi=150)
plt.show()

### Wykres 4 - Scatter: rzeczywiste vs przewidywane gazowanie 

Każdy panel = jeden podmiot. Punkty = timepoints, linia = regresja liniowa.
Tytuł panelu: $R^2$ dla tego podmiotu.

Wyniki: `figures\5-scatter_predicted_true_x.png`

Wyniki: `figures\5-scatter_predicted_true_y.png`

In [ ]:
for axis_name, pred_key, true_key, color in [
    ("X", "pred_x", "true_x", "#378ADD"),
    ("Y", "pred_y", "true_y", "#1D9E75"),
]:
    ncols = 6
    nrows = int(np.ceil(len(subjects) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3, nrows*3))
    axes = axes.flatten()

    for i, subj in enumerate(subjects):
        pred = all_preds_ws[subj][pred_key]
        true = all_preds_ws[subj][true_key]
        slope, intercept, r, _, _ = linregress(pred, true)
        r2 = r ** 2
        axes[i].scatter(pred, true, s=8, alpha=0.4, color=color)
        x_line = np.linspace(pred.min(), pred.max(), 100)
        axes[i].plot(x_line, slope * x_line + intercept, color="black", linewidth=1.5)
        axes[i].set_title(f"{subj[:4]}  R²={r2:.3f}", fontsize=9)
        axes[i].set_xlabel("predicted", fontsize=7)
        axes[i].set_ylabel("true", fontsize=7)
        axes[i].tick_params(labelsize=6)

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(f"{axis_name} gaze - true vs predicted (all subjects)", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"./plots/bottleneck_scatter_{axis_name.lower()}_all.png", dpi=150)
    plt.show()

## Dekodowanie międzypodmiotowe (across-subject)

Tutaj sprawdzamy, czy cechy wąskiego gardła są **generalizowalne między podmiotami**.

In [ ]:
X_all_x = np.vstack([data[s]["bottleneck_combined"][:, top200_x] for s in subjects])
X_all_y = np.vstack([data[s]["bottleneck_combined"][:, top200_y] for s in subjects])
y_all_x = np.concatenate([data[s]["true_x"] for s in subjects])
y_all_y = np.concatenate([data[s]["true_y"] for s in subjects])
groups = np.concatenate([np.full(182, i) for i, _ in enumerate(subjects)])

logo = LeaveOneGroupOut()

print(f"{'subj':<12} {'r_x':>8} {'r_y':>8}")

all_r_x_as, all_r_y_as = [], []
all_preds_as = {}

for train_idx, test_idx in logo.split(X_all_x, y_all_x, groups):
    subj_idx = groups[test_idx][0]
    subj = subjects[subj_idx]

    for axis, X_all, y_all, r_list, pred_key, true_key in [
        ("x", X_all_x, y_all_x, all_r_x_as, "pred_x", "true_x"),
        ("y", X_all_y, y_all_y, all_r_y_as, "pred_y", "true_y"),
    ]:
        X_train, X_test = X_all[train_idx], X_all[test_idx]
        y_train, y_test = y_all[train_idx], y_all[test_idx]

        signs = np.array([
            np.sign(pearsonr(X_train[:, f], y_train)[0])
            if X_train[:, f].std() > 1e-6 else 1.0
            for f in range(X_train.shape[1])
        ])
        signs[signs == 0] = 1.0
        X_train = X_train * signs
        X_test = X_test * signs

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        clf = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 1000, 10000])
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)

        r, _ = pearsonr(pred, y_test)
        r_list.append(r)

        if subj not in all_preds_as:
            all_preds_as[subj] = {}
        all_preds_as[subj][pred_key] = pred
        all_preds_as[subj][true_key] = y_test

    print(f"{subj:<12} {all_r_x_as[-1]:>8.3f} {all_r_y_as[-1]:>8.3f}")

print(f"\n{'mean':<12} {np.mean(all_r_x_as):>8.3f} {np.mean(all_r_y_as):>8.3f}")
print(f"{'std':<12} {np.std(all_r_x_as):>8.3f} {np.std(all_r_y_as):>8.3f}")
print(f"{'n>0.2':<12} {sum(r>0.2 for r in all_r_x_as):>8} {sum(r>0.2 for r in all_r_y_as):>8}")

### Wykres 5 - Pearson r per podmiot (across-subject)

Porównanie z dekodowaniem wewnątrzpodmiotowym:
generalnie wyniki across-subject są słabsze, ale pokazują
na ile wzorce cech są wspólne dla różnych podmiotów.

Wyniki: `figures\5-across_subject_r_per_subject.png`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(subjects))
labels = [s[:4] for s in subjects]

for ax, rs, title, base_color in [
    (axes[0], all_r_x_as, "X gaze — across-subject r", "#378ADD"),
    (axes[1], all_r_y_as, "Y gaze — across-subject r", "#1D9E75"),
]:
    colors = [base_color if r > 0 else "#E24B4A" for r in rs]
    ax.bar(x, rs, color=colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.2, color="red", linewidth=1, linestyle="--", label="r=0.2")
    ax.axhline(0.3, color="orange", linewidth=1, linestyle="--", label="r=0.3")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Pearson r")
    ax.set_ylim(-1, 1)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("./plots/bottleneck_across_subject_r.png", dpi=150)
plt.show()

### Wyniki $R^2$  dekodowanie międzypodmiotowe

Analogicznie do within-subject, obliczamy $R^2$ z regresji liniowej (true ~ pred).

In [ ]:
all_r2_x_as, all_r2_y_as, all_r2_xy_as = [], [], []

for subj in subjects:
    pred_x = all_preds_as[subj]["pred_x"]
    pred_y = all_preds_as[subj]["pred_y"]
    true_x = all_preds_as[subj]["true_x"]
    true_y = all_preds_as[subj]["true_y"]

    _, _, r_x, _, _ = linregress(pred_x, true_x)
    _, _, r_y, _, _ = linregress(pred_y, true_y)
    r2_x  = r_x ** 2
    r2_y  = r_y ** 2
    r2_xy = (r2_x + r2_y) / 2

    all_r2_x_as.append(r2_x)
    all_r2_y_as.append(r2_y)
    all_r2_xy_as.append(r2_xy)

print(f"mean R^2_x  = {np.mean(all_r2_x_as):.3f}")
print(f"mean R^2_y  = {np.mean(all_r2_y_as):.3f}")
print(f"mean R^2_xy = {np.mean(all_r2_xy_as):.3f}")

print(f"{'method':<22} {'mean_rx':>10} {'mean_ry':>10}")
print(f"{'within-subject':<22} {np.mean(all_r_x_ws):>10.3f} {np.mean(all_r_y_ws):>10.3f}")
print(f"{'across-subject (LOGO)':<22} {np.mean(all_r_x_as):>10.3f} {np.mean(all_r_y_as):>10.3f}")